# Import/Load Data


In [239]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

data_path = Path("../data/fraudTest.csv")
df = pd.read_csv(data_path)
print(list(df.columns))


df['is_fraud'].value_counts()









['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


is_fraud
0    553574
1      2145
Name: count, dtype: int64

# Feature engineering


In [ ]:

df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["dob"] = pd.to_datetime(df["dob"])

DROP_COLS = [
    "Unnamed: 0",
    "cc_num",
    "first",
    "last",
    "street",
    "zip",
    "trans_num",
    "unix_time",
    "merchant",   # IMPORTANT
    "job",
    "dob",
    "trans_date_trans_time"
]

df = df.drop(columns=DROP_COLS)

drop_cols = [
    "Unnamed: 0",
    "trans_date_trans_time",
    "dob",
    "trans_num",
    "cc_num",
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df.head()



In [243]:
TARGET = "is_fraud"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(X.columns)


Index(['category', 'amt', 'gender', 'city', 'state', 'lat', 'long', 'city_pop',
       'merch_lat', 'merch_long'],
      dtype='object')


# Train/Test Split

In [244]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)



In [245]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)



In [246]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logreg = LogisticRegression(
    class_weight="balanced",
    C=0.1,              # strong regularization
    max_iter=1000,
    solver="lbfgs"
)

pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", logreg)
])


In [247]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(
    estimator=pipeline,   # ✅ NEW name
    method="isotonic",
    cv=3
)

calibrated_model.fit(X_train, y_train)



CalibratedClassifierCV(cv=3,
                       estimator=Pipeline(steps=[('prep',
                                                  ColumnTransformer(transformers=[('num',
                                                                                   StandardScaler(),
                                                                                   Index(['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long'], dtype='object')),
                                                                                  ('cat',
                                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                                   Index(['category', 'gender', 'city', 'state'], dtype='object'))])),
                                                 ('clf',
                                                  LogisticRegression(C=0.1,
                                                                     class_weight='balanced',
                                                                     max_iter=1000))]),
                       method='isotonic')

In [248]:
from sklearn.metrics import roc_auc_score, classification_report

y_prob = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.3).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


ROC-AUC: 0.9749404985416367
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110715
           1       0.33      0.42      0.37       429

    accuracy                           0.99    111144
   macro avg       0.66      0.71      0.68    111144
weighted avg       1.00      0.99      0.99    111144



# Saving the practical model

In [249]:


from pathlib import Path
import joblib

MODEL_DIR = Path("/Users/ificouldcode/Desktop/Project/push2/credit_card_fd/models")

joblib.dump(calibrated_model, MODEL_DIR / "fraud_pipeline.pkl")




['/Users/ificouldcode/Desktop/Project/push2/credit_card_fd/models/fraud_pipeline.pkl']